# 6.3. 填充和步幅

在前面的例子中，我们的输入高度和宽度都为3，卷积核的高度和宽度都为2，得到的输出形状为2×2。一般来说，假设输入形状为 $n_h \times n_w$，卷积核形状为 $k_h \times k_w$，那么输出形状将是 $(n_h - k_h + 1) \times (n_w - k_w + 1)$。

因此，卷积层的输出形状取决于输入形状和卷积核形状。本节将介绍填充（padding）和步幅（stride），它们可以控制输出形状。

In [ ]:
import tensorflow as tf
import numpy as np

## 6.3.1. 填充

填充（padding）是在输入周围添加额外的行/列（通常填充0）。这样可以：
1. 保持输出大小与输入大小相同
2. 避免信息在边缘丢失过快

In [ ]:
# 定义一个计算卷积层的函数
def comp_conv2d(conv2d, X):
    # 这里的(1, 1)表示批量大小和通道数都是1
    X = tf.reshape(X, (1, ) + X.shape + (1, ))
    Y = conv2d(X)
    # 省略前两个维度：批量大小和通道
    return tf.reshape(Y, Y.shape[1:3])

In [ ]:
# 创建一个高度和宽度为3的二维卷积层，并在所有侧边填充1个像素
conv2d = tf.keras.layers.Conv2D(1, kernel_size=3, padding='same')
X = tf.random.uniform(shape=(8, 8))
output = comp_conv2d(conv2d, X)
print(f"输入形状: {X.shape}")
print(f"输出形状: {output.shape}")
print("填充后，输出形状与输入形状相同！")

In [ ]:
# 使用高度为5、宽度为3的卷积核，高度和宽度两边的填充分别为2和1
conv2d = tf.keras.layers.Conv2D(1, kernel_size=(5, 3), padding='same')
output = comp_conv2d(conv2d, X)
print(f"输入形状: {X.shape}")
print(f"输出形状: {output.shape}")

## 6.3.2. 步幅

步幅（stride）是指滑动窗口每次移动的行数和列数。

In [ ]:
# 将高度和宽度的步幅设置为2
conv2d = tf.keras.layers.Conv2D(1, kernel_size=3, padding='same', strides=2)
output = comp_conv2d(conv2d, X)
print(f"输入形状: {X.shape}")
print(f"输出形状: {output.shape}")
print("步幅为2时，输出的高度和宽度都减半！")

In [ ]:
# 使用更复杂的例子
conv2d = tf.keras.layers.Conv2D(
    1, kernel_size=(3, 5), padding='valid', strides=(3, 4))
output = comp_conv2d(conv2d, X)
print(f"输入形状: {X.shape}")
print(f"输出形状: {output.shape}")

## 6.3.3. 实际应用

在实践中：
- **填充** 通常用于保持空间维度
- **步幅** 通常用于下采样（减小空间维度）

In [ ]:
# 常见的卷积配置
print("常见配置示例：")
print()

# 1. 保持尺寸：3x3卷积，填充1，步幅1
conv1 = tf.keras.layers.Conv2D(64, kernel_size=3, padding='same', strides=1)
X1 = tf.random.uniform((1, 32, 32, 3))  # 批量大小、高、宽、通道
Y1 = conv1(X1)
print(f"配置1 - 保持尺寸: {X1.shape} -> {Y1.shape}")

# 2. 下采样：3x3卷积，填充1，步幅2
conv2 = tf.keras.layers.Conv2D(64, kernel_size=3, padding='same', strides=2)
Y2 = conv2(X1)
print(f"配置2 - 下采样2倍: {X1.shape} -> {Y2.shape}")

# 3. 下采样：5x5卷积，填充2，步幅2
conv3 = tf.keras.layers.Conv2D(128, kernel_size=5, padding='same', strides=2)
Y3 = conv3(X1)
print(f"配置3 - 下采样2倍（大核）: {X1.shape} -> {Y3.shape}")

## 小结

1. **填充**可以增加输出的高度和宽度，常用来使输出与输入具有相同的高和宽
2. **步幅**可以减小输出的高度和宽度，例如输出的高和宽仅为输入的1/2（步幅为2时）
3. 填充和步幅可用于有效地调整数据的维度
4. 常见做法：
   - 使用奇数核大小（如3×3、5×5），配合适当填充保持尺寸
   - 使用步幅2进行下采样，减少计算量